# LeetCode 3177 — Find the Maximum Length of a Good Subsequence II bằng Python

- Link: https://leetcode.com/problems/find-the-maximum-length-of-a-good-subsequence-ii/
- Độ khó: **Hard**
- Chủ đề: **DP dãy con theo (giá trị cuối, số lần đổi)** — `f[x][j]` = độ dài dài nhất kết thúc bằng giá trị `x` với `≤ j` lần đổi; `g[j] = max_x f[x][j]`; mỗi phần tử cập nhật `O(k)` ⇒ `O(n · k)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumLength(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu "dãy tốt": số vị trí kề nhau khác nhau `≤ k` — tức dãy gồm `≤ k + 1` **khối** giá trị liên tiếp bằng nhau.
2. DP `O(n² k)` theo chỉ số kết thúc (bản I): `dp[i][j] = 1 + max(dp[p][j] (cùng giá trị), dp[p][j − 1] (khác giá trị))`.
3. Tối ưu: thay `max` theo `p` bằng hai bảng chạy — `f[x][j]` (cùng giá trị `x`) và `g[j]` (bất kỳ) ⇒ `O(n · k)`.
4. Thứ tự cập nhật `j` **giảm dần** để không dùng chính phần tử hiện tại hai lần.
5. Kiểm chứng bằng brute liệt kê dãy con.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `k ≥ 0`. Dãy `seq` là **tốt** nếu có **nhiều nhất `k`** chỉ số `i ∈ [0, len − 2]` với `seq[i] ≠ seq[i + 1]`. Tìm độ dài lớn nhất của một **dãy con** (giữ thứ tự, không cần liên tiếp) tốt của `nums`.

**Ví dụ 1:** `nums = [1,2,1,1,3]`, `k = 2` → `4` (ví dụ `[1, 1, 1, 3]` hoặc `[1, 2, 1, 1]`).

**Ví dụ 2:** `nums = [1,2,3,4,5,1]`, `k = 0` → `2` (`[1, 1]`).

**Ràng buộc:** `1 ≤ n ≤ 5·10³`; `1 ≤ nums[i] ≤ 10⁹`; `0 ≤ k ≤ min(50, n)`.

In [1]:
from typing import List
from collections import defaultdict
import itertools, random, time

examples = [([1, 2, 1, 1, 3], 2, 4), ([1, 2, 3, 4, 5, 1], 0, 2)]
for a, k, exp in examples:
    print(a, 'k =', k, '-> kỳ vọng', exp)

[1, 2, 1, 1, 3] k = 2 -> kỳ vọng 4
[1, 2, 3, 4, 5, 1] k = 0 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Đếm số lần đổi của một dãy

In [2]:
def changes(seq):
    return sum(1 for i in range(len(seq) - 1) if seq[i] != seq[i + 1])

for s in ([1, 1, 1, 3], [1, 2, 1, 1], [1, 2, 1, 1, 3], [7], []):
    print(s, '-> số lần đổi =', changes(s))

[1, 1, 1, 3] -> số lần đổi = 1
[1, 2, 1, 1] -> số lần đổi = 2
[1, 2, 1, 1, 3] -> số lần đổi = 3
[7] -> số lần đổi = 0
[] -> số lần đổi = 0


### 3.2. DP "kết thúc tại đâu, đã dùng bao nhiêu"

Khi nối phần tử `x` vào cuối một dãy tốt kết thúc bằng `y`: nếu `y = x` số lần đổi giữ nguyên, nếu `y ≠ x` tăng 1. Vậy trạng thái chỉ cần **giá trị cuối** và **số lần đổi đã dùng** — không cần biết toàn bộ dãy.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đọc `nums` từ trái sang phải. Với mỗi số lần đổi `j`, nhớ:

- `f[x][j]`: dãy tốt dài nhất **kết thúc bằng giá trị `x`**, dùng `≤ j` lần đổi;
- `g[j]`: dãy tốt dài nhất (kết thúc bằng bất kỳ giá trị nào), dùng `≤ j` lần đổi.

Gặp `x`: hoặc nối vào dãy đang kết thúc bằng `x` (không tốn), hoặc nối vào dãy tốt nhất bất kỳ với `j − 1` lần đổi (tốn 1 — nếu dãy đó kết thúc bằng `x` thì thực ra không tốn, vẫn hợp lệ vì "nhiều nhất").

## 5. Cách 1 — Brute Force: liệt kê mọi dãy con

In [3]:
def brute(nums, k):
    n = len(nums); best = 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if changes(seq) <= k:
            best = max(best, len(seq))
    return best

for a, k, exp in examples:
    print('brute:', brute(a, k), '| kỳ vọng', exp)

brute: 4 | kỳ vọng 4
brute: 2 | kỳ vọng 2


**Độ phức tạp:** `O(2^n · n)` — chỉ cho `n ≤ 14`.

## 6. Ý tưởng thuật toán / bước đột phá

```
f = {}                               # f[x] = [độ dài tốt nhất với ≤ j lần đổi, j = 0..k]
g = [0] * (k + 1)
for x in nums:
    fx = f.setdefault(x, [0] * (k + 1))
    for j = k .. 0 (giảm dần):
        fx[j] = max(fx[j] + 1, (g[j-1] + 1) nếu j > 0)
        g[j]  = max(g[j], fx[j])
return g[k]
```

`O(n · k)` = `5000 · 51 ≈ 2.6·10⁵` bước.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo chỉ số kết thúc `O(n² · k)` (bản I)

In [4]:
def dp_n2k(nums, k):
    n = len(nums); dp = [[1] * (k + 1) for _ in range(n)]; best = 1
    for i in range(n):
        for p in range(i):
            for j in range(k + 1):
                if nums[p] == nums[i]:
                    dp[i][j] = max(dp[i][j], dp[p][j] + 1)
                elif j > 0:
                    dp[i][j] = max(dp[i][j], dp[p][j - 1] + 1)
        best = max(best, dp[i][k])
    return best

for a, k, exp in examples:
    print('O(n²k):', dp_n2k(a, k), '| kỳ vọng', exp)

O(n²k): 4 | kỳ vọng 4
O(n²k): 2 | kỳ vọng 2


### 7.2. Cách 3 — Hai bảng chạy `f[x][j]`, `g[j]` (lời giải chính)

In [5]:
def running_tables(nums, k):
    f = {}; g = [0] * (k + 1)
    for x in nums:
        fx = f.get(x)
        if fx is None:
            fx = f[x] = [0] * (k + 1)
        for j in range(k, -1, -1):
            v = fx[j] + 1
            if j and g[j - 1] + 1 > v:
                v = g[j - 1] + 1
            fx[j] = v
            if v > g[j]:
                g[j] = v
    return g[k]

for a, k, exp in examples:
    print('f/g:', running_tables(a, k), '| kỳ vọng', exp)

f/g: 4 | kỳ vọng 4
f/g: 2 | kỳ vọng 2


### 7.3. Cách 4 — DP theo tầng `j` (tính cả mảng cho từng số lần đổi)

Tầng `j` tính từ tầng `j − 1`: `dp_j[i] = 1 + max( max_{p<i, nums[p]=nums[i]} dp_j[p], max_{p<i} dp_{j−1}[p] )`. Duyệt `i` tăng dần, giữ **max tiền tố** của tầng trước và **max theo giá trị** của tầng hiện tại.

In [6]:
def by_layers(nums, k):
    n = len(nums)
    prev = None
    for j in range(k + 1):
        cur = [0] * n; by_val = {}; run_prev = 0
        for i, x in enumerate(nums):
            best = by_val.get(x, 0)
            if prev is not None:
                best = max(best, run_prev)
            cur[i] = best + 1
            if cur[i] > by_val.get(x, 0):
                by_val[x] = cur[i]
            if prev is not None and prev[i] > run_prev:
                run_prev = prev[i]                     # max tiền tố tầng j−1 tới i (dùng cho i+1..)
        prev = cur
    return max(prev)

for a, k, exp in examples:
    print('theo tầng:', by_layers(a, k), '| kỳ vọng', exp)

theo tầng: 4 | kỳ vọng 4
theo tầng: 2 | kỳ vọng 2


Lưu ý thứ tự: `run_prev` chỉ được cập nhật bằng `prev[i]` **sau** khi tính `cur[i]` (phải là `p < i`).

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 (`k = 2`): in `f[x]` và `g` sau mỗi phần tử.

In [7]:
def trace(nums, k):
    f = {}; g = [0] * (k + 1)
    for i, x in enumerate(nums):
        fx = f.setdefault(x, [0] * (k + 1))
        for j in range(k, -1, -1):
            fx[j] = max(fx[j] + 1, g[j - 1] + 1 if j else 0)
            g[j] = max(g[j], fx[j])
        print(f'i = {i}, x = {x}: f = { {v: lst for v, lst in f.items()} } | g = {g}')
    print('đáp án g[k] =', g[k], '| brute:', brute(nums, k))

trace(*examples[0][:2])

i = 0, x = 1: f = {1: [1, 1, 1]} | g = [1, 1, 1]
i = 1, x = 2: f = {1: [1, 1, 1], 2: [1, 2, 2]} | g = [1, 2, 2]
i = 2, x = 1: f = {1: [2, 2, 3], 2: [1, 2, 2]} | g = [2, 2, 3]
i = 3, x = 1: f = {1: [3, 3, 4], 2: [1, 2, 2]} | g = [3, 3, 4]
i = 4, x = 3: f = {1: [3, 3, 4], 2: [1, 2, 2], 3: [1, 4, 4]} | g = [3, 4, 4]
đáp án g[k] = 4 | brute: 4


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maximumLength(self, nums: List[int], k: int) -> int:
        f = {}
        g = [0] * (k + 1)
        for x in nums:
            fx = f.get(x)
            if fx is None:
                fx = [0] * (k + 1)
                f[x] = fx
            for j in range(k, -1, -1):
                v = fx[j] + 1
                if j and g[j - 1] + 1 > v:
                    v = g[j - 1] + 1
                fx[j] = v
                if v > g[j]:
                    g[j] = v
        return g[k]

for a, k, exp in examples:
    got = Solution().maximumLength(a, k)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Bất biến** (sau khi xử lý tiền tố `nums[0..i]`): `f[x][j]` = độ dài lớn nhất của dãy con tốt **kết thúc bằng giá trị `x`** với `≤ j` lần đổi; `g[j] = max_x f[x][j]`.
2. **Chuyển:** dãy tối ưu mới kết thúc tại phần tử `x` hiện tại là `x` nối sau một dãy (có thể rỗng) kết thúc bằng `y`. Nếu `y = x`: độ dài `f[x][j] + 1` (`f` cũ). Nếu `y ≠ x`: cần dãy có `≤ j − 1` lần đổi ⇒ `≤ g[j − 1] + 1`; ngược lại mọi dãy đạt `g[j − 1]` nối `x` vào có `≤ j` lần đổi ⇒ hợp lệ. Dãy rỗng + `x` có độ dài 1 `≤ f[x][j] + 1`.
3. **Thứ tự `j` giảm dần:** khi tính tầng `j` ta đọc `g[j − 1]` **chưa** được cập nhật bởi phần tử hiện tại ⇒ không nối `x` vào chính nó.
4. `g[k]` là đáp án vì "nhiều nhất `k`" đã gộp mọi `j ≤ k`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(2^n · n)` | `O(n)` |
| 2. DP chỉ số | `O(n² · k)` | `O(n · k)` |
| 3. Bảng chạy `f`, `g` | `O(n · k)` | `O(D · k)` (`D` giá trị khác nhau) |
| 4. Theo tầng | `O(n · k)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'n2k': dp_n2k, 'running': running_tables, 'layers': by_layers,
         'submit': lambda a, k: Solution().maximumLength(a, k)}
edge = [([5], 0, 1), ([5], 3, 1), ([1, 2, 3], 2, 3), ([1, 2, 3], 1, 2), ([1, 2, 1, 2, 1, 2], 1, 4),
        ([1, 2, 1, 2, 1, 2], 0, 3), ([3, 3, 3, 3], 0, 4), ([1, 2, 1, 3, 1], 2, 4)]
for a, k, exp in examples + edge:
    assert brute(a, k) == exp, (a, k, brute(a, k))
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 2, 1, 2, 1, 2]`, `k = 1` — hai khối: ba số `1` rồi số `2` cuối (`1, 1, 1, 2`) ⇒ `4`. `[1, 2, 1, 3, 1]`, `k = 2`: `1, 1, 3, 1`? có 2 lần đổi ⇒ `4`; hoặc `1, 2, 1, 1`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3177)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 12); a = [rng.randint(1, rng.choice([2, 3, 5])) for _ in range(n)]
    k = rng.randint(0, min(4, n))
    exp = brute(a, k)
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 400 bộ ngẫu nhiên.


In [11]:
# n vừa: so với DP O(n²k)
for _ in range(20):
    n = rng.randint(20, 120); a = [rng.randint(1, 10) for _ in range(n)]; k = rng.randint(0, 10)
    assert running_tables(a, k) == by_layers(a, k) == dp_n2k(a, k) == Solution().maximumLength(a, k)
print('Các cách O(nk) khớp DP O(n²k) với n ≤ 120.')

Các cách O(nk) khớp DP O(n²k) với n ≤ 120.


### 12.3. Ca lớn `n = 5000`, `k = 50`

In [12]:
N = 5000
big = {
    'ngẫu nhiên lớn': ([rng.randint(1, 10 ** 9) for _ in range(N)], 50),
    'ít giá trị': ([rng.randint(1, 5) for _ in range(N)], 50),
    'xen kẽ 1,2': ([1 + i % 2 for i in range(N)], 50),
}
for name, (a, k) in big.items():
    t = time.time(); x = Solution().maximumLength(a, k); t1 = time.time() - t
    t = time.time(); y = by_layers(a, k); t2 = time.time() - t
    assert x == y
    print(f'{name:16s}: {x:5d} | nộp {t1:.2f}s | theo tầng {t2:.2f}s')
assert Solution().maximumLength([rng.randint(1, 10 ** 9) for _ in range(N)], 50) >= 51
print('Giá trị ngẫu nhiên lớn: ít nhất k + 1 = 51 phần tử (mỗi khối một phần tử).')

ngẫu nhiên lớn  :    51 | nộp 0.03s | theo tầng 0.04s


ít giá trị      :  1481 | nộp 0.02s | theo tầng 0.05s


xen kẽ 1,2      :  2525 | nộp 0.02s | theo tầng 0.05s
Giá trị ngẫu nhiên lớn: ít nhất k + 1 = 51 phần tử (mỗi khối một phần tử).


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `k + 1` tần suất lớn nhất (bỏ qua thứ tự)

In [13]:
from collections import Counter
def wrong_top_freq(nums, k):
    return sum(c for _, c in Counter(nums).most_common(k + 1))          # SAI

args = ([1, 2, 1, 2, 1, 2], 1)
print('sai :', wrong_top_freq(*args), '| đúng:', Solution().maximumLength(*args))

sai : 6 | đúng: 4


**Vì sao sai:** các số `1` và `2` đan xen; dãy con chỉ có 2 khối nên không lấy hết được. Chỉ đúng khi `k = 0` (đáp án = tần suất lớn nhất).

### Lỗi 2 — Duyệt `j` **tăng dần** trong cập nhật tại chỗ

In [14]:
def wrong_ascending_j(nums, k):
    f = {}; g = [0] * (k + 1)
    for x in nums:
        fx = f.setdefault(x, [0] * (k + 1))
        for j in range(k + 1):                                          # SAI: tăng dần
            fx[j] = max(fx[j] + 1, g[j - 1] + 1 if j else 0)
            g[j] = max(g[j], fx[j])
    return g[k]

print('sai :', wrong_ascending_j([7], 1), '| đúng:', Solution().maximumLength([7], 1))

sai : 2 | đúng: 1


**Vì sao sai:** ở `j = 1` ta đọc `g[0]` đã chứa **chính** phần tử `7` vừa được tính ở `j = 0` ⇒ nối `7` vào sau chính nó, độ dài 2 từ một phần tử.

### Lỗi 3 — Hiểu nhầm "≤ k lần đổi" thành "≤ k + 1 giá trị khác nhau"

In [15]:
def wrong_distinct(nums, k):
    vals = sorted(set(nums)); best = 0
    for r in range(1, min(len(vals), k + 1) + 1):
        for S in itertools.combinations(vals, r):
            best = max(best, sum(1 for x in nums if x in S))             # SAI
    return best

args = ([1, 2, 1], 1)
print('sai :', wrong_distinct(*args), '| đúng:', Solution().maximumLength(*args))

sai : 3 | đúng: 2


**Vì sao sai:** `[1, 2, 1]` chỉ dùng 2 giá trị nhưng có 2 lần đổi (`1→2`, `2→1`). Giá trị có thể **quay lại** nhưng mỗi lần quay lại tốn một lần đổi.

### Lỗi 4 — Chỉ cho phép đổi khi nối vào dãy tốt nhất (quên nhánh "cùng giá trị")

In [16]:
def wrong_no_same(nums, k):
    g = [0] * (k + 1)
    for x in nums:
        for j in range(k, -1, -1):
            g[j] = max(g[j], (g[j - 1] + 1) if j else 1)                 # SAI: mọi lần nối đều tốn 1 lần đổi
    return g[k]

print('sai :', wrong_no_same([3, 3, 3, 3], 0), '| đúng:', Solution().maximumLength([3, 3, 3, 3], 0))

sai : 1 | đúng: 4


**Vì sao sai:** nối `x` sau một dãy kết thúc bằng `x` **không** tốn lần đổi — cần bảng `f[x][j]` riêng theo giá trị cuối.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Dãy con, ràng buộc trên số lần **giá trị kề nhau khác nhau**" ⇒ trạng thái (giá trị cuối, số lần đổi).
- `max` trên các phần tử trước có cùng giá trị ⇒ từ điển theo giá trị; `max` trên tất cả ⇒ một mảng `g`.
- Cập nhật tại chỗ với tầng phụ thuộc tầng thấp hơn ⇒ duyệt tầng **giảm dần** (giống knapsack 0/1).

## 15. Biến thể của bài

1. **LC 3176 (bản I):** `n ≤ 500`, `k ≤ 25` — DP `O(n²k)` đủ.
2. Đổi "nhiều nhất `k`" thành "**đúng** `k`" lần đổi ⇒ không gộp tầng, thêm `-∞` cho trạng thái không đạt.
3. Mỗi lần đổi có **chi phí** phụ thuộc cặp giá trị ⇒ cần trạng thái tốn kém hơn.
4. Chuỗi ký tự: "dãy con dài nhất có ≤ `k` lần đổi ký tự" ⇒ `f` là mảng 26 phần tử.

In [17]:
# Biến thể 2: đúng k lần đổi — cần max trên các giá trị cuối KHÁC x ⇒ giữ top-2 (giá trị cuối khác nhau) mỗi tầng
NEG = float('-inf')
def exactly_k(nums, k):
    f = {}
    top = [[(NEG, None), (NEG, None)] for _ in range(k + 1)]      # (độ dài, giá trị cuối), hai giá trị khác nhau
    def best_other(j, x):
        (l1, v1), (l2, v2) = top[j]
        return l1 if v1 != x else l2
    def push(j, length, x):
        (l1, v1), (l2, v2) = top[j]
        if v1 == x:
            top[j][0] = (max(l1, length), x)
        elif length > l1:
            top[j] = [(length, x), (l1, v1)]
            return
        elif v2 == x or length > l2:
            top[j][1] = (max(length, l2 if v2 == x else NEG), x)
        if top[j][1][0] > top[j][0][0]:
            top[j] = [top[j][1], top[j][0]]
    for x in nums:
        fx = f.setdefault(x, [NEG] * (k + 1))
        for j in range(k, -1, -1):
            v = max(fx[j] + 1, 1 if j == 0 else NEG, best_other(j - 1, x) + 1 if j else NEG)
            fx[j] = v
            push(j, v, x)
    return max((lst[k] for lst in f.values()), default=NEG) if any(lst[k] > NEG for lst in f.values()) else -1

def exactly_k_brute(nums, k):
    n = len(nums); best = -1
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if changes(seq) == k:
            best = max(best, len(seq))
    return best

for _ in range(200):
    a = [rng.randint(1, 3) for _ in range(rng.randint(1, 10))]; k = rng.randint(0, 4)
    assert exactly_k(a, k) == exactly_k_brute(a, k), (a, k)
print('Biến thể "đúng k lần đổi": khớp brute.')

Biến thể "đúng k lần đổi": khớp brute.


**Lưu ý:** với "đúng `k`", nhánh đổi giá trị phải lấy dãy trước kết thúc bằng giá trị **khác** `x` (nếu nó kết thúc bằng `x`, nối vào chỉ tạo `j − 1` lần đổi). Vì vậy `g[j]` được thay bằng **top-2** theo giá trị cuối khác nhau: nếu giá trị tốt nhất trùng `x` thì lấy giá trị nhì. Với "nhiều nhất `k`" (bài gốc) không cần top-2 vì nối nhầm chỉ làm *ít* lần đổi hơn — vẫn hợp lệ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê dãy con | `O(2^n n)` | Oracle |
| 2 | DP theo chỉ số kết thúc | `O(n²k)` | Bản I |
| 3 | Bảng `f[x][j]`, `g[j]` | `O(nk)` | **Nộp bài** |
| 4 | Theo tầng `j` | `O(nk)` | Bộ nhớ `O(n)` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [4, 4, 5, 4]`, `k = 0`.

**Bài 2 (tính tay):** `nums = [1, 2, 3, 1, 2, 3]`, `k = 1`.

**Bài 3 (điền TODO):** trả về **một** dãy con tốt dài nhất (danh sách chỉ số).

**Bài 4 (tự viết lại):** với mỗi `k' = 0..k`, trả về độ dài lớn nhất (mảng `k + 1` phần tử) trong một lần chạy.

**Bài 5 (mini challenge):** đếm số **chỉ số** `k` nhỏ nhất để có thể lấy **toàn bộ** mảng (tức số lần đổi của chính `nums`) — và chứng minh `g[k] = n` ⇔ `k ≥ changes(nums)`.

In [18]:
def reconstruct_todo(nums, k):
    # TODO
    raise NotImplementedError

def all_k_todo(nums, k):
    # TODO
    raise NotImplementedError

def min_k_for_all_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 0` ⇒ một khối duy nhất. (2) Tần suất lớn nhất. (3) `3`.

**Bài 2.** (1) Hai khối. (2) Thử `1, 1` + `2` hoặc `1` + `2, 2`… (3) `3` (ví dụ `1, 1, 2` hay `1, 2, 2`) — kiểm bằng brute.

**Bài 3.** (1) Dùng DP `O(n²k)` có mảng cha `par[i][j]`. (2) Chọn `i` có `dp[i][k]` lớn nhất. (3) Truy ngược: nếu cha cùng giá trị giữ `j`, khác giá trị giảm `j`.

**Bài 4.** (1) `g` sau khi chạy xong chính là mảng cần tìm. (2) Trả `g[:]`. (3) Kiểm từng `k'` bằng lời giải đơn.

**Bài 5.** (1) Lấy toàn bộ mảng tốn đúng `changes(nums)` lần đổi. (2) `g[k] ≤ n` luôn; `= n` cần dùng mọi phần tử ⇒ dãy chính là `nums`. (3) Đáp án `changes(nums)`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 4, 5
assert Solution().maximumLength([4, 4, 5, 4], 0) == 3
ans2 = Solution().maximumLength([1, 2, 3, 1, 2, 3], 1)
assert ans2 == brute([1, 2, 3, 1, 2, 3], 1) == 3

def all_k(nums, k):
    f = {}; g = [0] * (k + 1)
    for x in nums:
        fx = f.setdefault(x, [0] * (k + 1))
        for j in range(k, -1, -1):
            fx[j] = max(fx[j] + 1, g[j - 1] + 1 if j else 0); g[j] = max(g[j], fx[j])
    return g[:]

def min_k_for_all(nums):
    return changes(nums)

for _ in range(150):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]; k = rng.randint(0, 5)
    assert all_k(a, k) == [Solution().maximumLength(a, kk) for kk in range(k + 1)]
    m = min_k_for_all(a)
    assert Solution().maximumLength(a, m) == len(a) and (m == 0 or Solution().maximumLength(a, m - 1) < len(a))
print('Bài 1: 3 | Bài 2:', ans2, '| Bài 4: mảng g khớp từng k | Bài 5: k nhỏ nhất = changes(nums).')

Bài 1: 3 | Bài 2: 3 | Bài 4: mảng g khớp từng k | Bài 5: k nhỏ nhất = changes(nums).


In [20]:
# Bài 3: khôi phục một dãy con tối ưu (DP O(n²k) có cha)
def reconstruct(nums, k):
    n = len(nums)
    dp = [[1] * (k + 1) for _ in range(n)]; par = [[None] * (k + 1) for _ in range(n)]
    for i in range(n):
        for p in range(i):
            for j in range(k + 1):
                if nums[p] == nums[i] and dp[p][j] + 1 > dp[i][j]:
                    dp[i][j] = dp[p][j] + 1; par[i][j] = (p, j)
                elif nums[p] != nums[i] and j > 0 and dp[p][j - 1] + 1 > dp[i][j]:
                    dp[i][j] = dp[p][j - 1] + 1; par[i][j] = (p, j - 1)
    i = max(range(n), key=lambda t: dp[t][k]); j = k; idx = []
    while i is not None:
        idx.append(i)
        nxt = par[i][j]
        if nxt is None:
            break
        i, j = nxt
    return idx[::-1]

for _ in range(200):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]; k = rng.randint(0, 4)
    idx = reconstruct(a, k)
    assert idx == sorted(idx) and len(set(idx)) == len(idx)
    seq = [a[i] for i in idx]
    assert changes(seq) <= k and len(seq) == Solution().maximumLength(a, k)
print('Bài 3: dãy con khôi phục hợp lệ và dài nhất; ví dụ 1 ->', reconstruct(*examples[0][:2]))

Bài 3: dãy con khôi phục hợp lệ và dài nhất; ví dụ 1 -> [0, 1, 2, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = {}; g = [0] * (k + 1)
for x in nums:
    fx = f.setdefault(x, [0] * (k + 1))
    for j in range(k, -1, -1):
        fx[j] = max(fx[j] + 1, g[j - 1] + 1 if j else 0)
        g[j] = max(g[j], fx[j])
return g[k]
```

**Checklist:**

- [ ] Hai nhánh: cùng giá trị (`f[x][j] + 1`) và khác giá trị (`g[j − 1] + 1`)?
- [ ] Duyệt `j` **giảm dần**?
- [ ] Từ điển theo giá trị (giá trị tới `10⁹` — không dùng mảng chỉ số theo giá trị)?
- [ ] Trả `g[k]` ("nhiều nhất `k`")?